# Load Packages

In [1]:
from typing import List, Tuple, Union, Any
import os
import torch
from transformers import PreTrainedTokenizer, AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm
from IPython.display import clear_output
import numpy as np
import json
import time
import re

os.environ["TOKENIZERS_PARALLELISM"] = "true"

In [2]:
from utils.llm import get_torch_device, generate_llm_tokens, unwatermarked_token_generation
from utils.generic import normalize_name, convert_token_func_to_intervals
from watermarking.tokens import (
    gumbel_token_generation,
    inverse_token_generation,
    pf_token_generation,
    redgreen_token_generation,
)
from watermarking.pivots import (
    pivot_statistic_gumbel_func,
    pivot_statistic_inverse_func,
    pivot_statistic_pf_func,
    pivot_statistic_redgreen_func,
)

# Some Utility Functions and Loading prompts

In [3]:
root_data_path = ".."
output_data_path = "../data"

# Read the list of prompts
def get_prompts():
    with open(os.path.join(root_data_path, "data/prompts_subset.txt"), "r", errors="ignore") as f:
        prompts = f.read().split("\n===\n")
        f.close()
    return prompts

def normalize_name(name: str):
    name = re.sub(r'[^A-Za-z0-9]', '-', name)
    name = re.sub(r'-+', '-', name)
    return name

prompt_list = get_prompts()
print(prompt_list[0][:500])

The Mapes family of Effingham enjoy the Lincoln Park Zoo in Chicago with their children including their adopted children, Regino and Regina, who were born in the Philippines.
Misty Mapes and her husband, Patrick, of Effingham always had a desire to add to their family through adoption.
That dream became a reality in part due to Gift of Adoption Fund, a nonprofit organization that provides financial support to families that need help to pay for the hefty cost of adopting a child.
The Mapes, who h


In [4]:
def generate_watermarked_data(
    model_name: str,
    token_generation_func: dict,
    pivot_func: Any = None,
    device: Any = None,
    output_filename: Union[str, None] = None,
    prompt_tokens: int = 50,
    output_tokens: int = 200,
    batch_size: int = 8,
    max_token_input_length: int = 256,
    initial_seed: int = 1234
):
    if device is None:
        device = get_torch_device(force_cpu=True)

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device) # type: ignore
    vocab_size = model.get_output_embeddings().weight.shape[0]
    print(f"There are {vocab_size} many words in vocabulary")
    print(f"The model {model_name} is loaded on device: {device}")

    # calculate the intervals
    intervals = []
    last_interval_type = None
    last_interval_index = None
    data_gen_type = "unwatermarked"
    for index in sorted([int(x) for x in token_generation_func.keys()], reverse = False):
        if last_interval_type is not None:
            intervals.append((last_interval_index, index, last_interval_type))
        last_interval_type = token_generation_func[str(index)].__name__.split('_')[0]
        if last_interval_type != "unwatermarked":
            data_gen_type = last_interval_type
        last_interval_index = index
    intervals.append((last_interval_index, output_tokens, last_interval_type))

    data_out_conf = {
        "model_name": model_name,
        "intervals": intervals,
        "prompt_tokens": prompt_tokens,
        "out_tokens": output_tokens,
        "vocab_size": vocab_size,
        "initial_seed": initial_seed,
        "max_token_input_length": max_token_input_length
    }
    if output_filename is None:
        output_filename = f"data_{normalize_name(model_name)}_n{output_tokens}_{data_gen_type}.json"

    response_list = []
    pivot_seed = initial_seed + prompt_tokens
    prompt_list = get_prompts()

    # progress initialize
    print(f"Starting loop for {int(len(prompt_list) / batch_size)} batches. Current time: {time.ctime()}")

    for i in tqdm(range(0, len(prompt_list), batch_size), desc = "Processing batches..."):
        prompt_batch = prompt_list[i:(i+batch_size)]
        response = generate_llm_tokens(
            prompt_batch,
            tokenizer,
            model,
            token_generation_func=token_generation_func,
            verbose=False,
            out_tokens=output_tokens,
            prompt_tokens=prompt_tokens,
            vocab_size=vocab_size,
            max_token_input_length=max_token_input_length,
            batch_size = batch_size
        )
        if pivot_func is not None:
            # calculate pivot function as well
            for j in range(len(response)):
                gen_tokens = response[j]["gen_tokens"]
                response[j]["pivots"] = pivot_func(gen_tokens, seed = pivot_seed, vocab_size = vocab_size)
        response_list.extend(response)

        # progress printing
        # print(f"Completed batch {i + 1} out of {int(len(prompt_list) / batch_size)} batches. Current time: {time.ctime()}")

        # save the json file
        with open(os.path.join(output_data_path, output_filename), "w") as f:
            json.dump({"configuration": data_out_conf, "data": response_list}, f)
            f.close()

    # save it at last as well
    with open(os.path.join(output_data_path, output_filename), "w") as f:
        json.dump({"configuration": data_out_conf, "data": response_list}, f)
        f.close()

# Run the data generation process

In [5]:
device = get_torch_device()
# torch.set_num_threads(8) # parallelize with 8 threads max

output_tokens = 2500
# model_name = "facebook/opt-125m"
# model_name = "google/gemma-3-270m"
model_name = "facebook/opt-1.3b"
# model_name = "princeton-nlp/Sheared-LLaMA-1.3B"
# model_name = "mistralai/Mistral-7B-v0.1"
# model_name = "meta-llama/Meta-Llama-3-8B"
# wm_methods = [gumbel_token_generation, inverse_token_generation, redgreen_token_generation, pf_token_generation]
# wm_pivots = [pivot_statistic_gumbel_func, pivot_statistic_inverse_func, pivot_statistic_redgreen_func, pivot_statistic_pf_func]
wm_methods = [gumbel_token_generation]
wm_pivots = [pivot_statistic_gumbel_func]


for wm_method, wm_pivot_func in zip(wm_methods, wm_pivots):
    print(f"\n\nRunning for watermarking method: {wm_method.__name__}")
    token_generation_func = {
        "0": unwatermarked_token_generation,  # (0-100) unwatermarked
        "100": wm_method,                     # (100-200) watermarke
        "200": unwatermarked_token_generation,  # (200-350) unwa
        "350": wm_method,                     # (350-500) water
        "500": unwatermarked_token_generation, # (500-700) unwat
        "700": wm_method,                     # (700-900) waterm
        "900": unwatermarked_token_generation,  # (900-1150) unwater
        "1150": wm_method,                    # (1150-1400) water
        "1400": unwatermarked_token_generation, # (1400-1700) unwater
        "1700": wm_method,                    # (1700-2000) watermarked
        "2000": unwatermarked_token_generation
    }
    pivot_func = wm_pivot_func
    # pivot_func = None

    # generate data
    generate_watermarked_data(
        model_name,
        token_generation_func,
        pivot_func,
        output_tokens=output_tokens,
        device=device,
        batch_size=2
    )



Running for watermarking method: gumbel_token_generation
There are 50272 many words in vocabulary
The model facebook/opt-1.3b is loaded on device: mps
Starting loop for 100 batches. Current time: Sat Mar 14 16:10:17 2026


Processing batches...:   0%|          | 0/100 [00:00<?, ?it/s]

KeyboardInterrupt: 